### Learning Goal
- **What concept is this notebook teaching?** Item-Based Collaborative Filtering using k-Nearest Neighbors (kNN).
- **Why does it matter?** It demonstrates how to build a basic recommendation engine by representing movies as vectors of user ratings and calculating the Cosine Similarity between them.

In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv('./data/ml-latest-small/movies.csv')
ratings = pd.read_csv('./data/ml-latest-small/ratings.csv')

print(movies.head())
print(ratings.head())

In [ ]:
movie_data = pd.merge(ratings, movies, on='movieId')

ratings_mean_count = pd.DataFrame(movie_data.groupby('title')['rating'].mean())
ratings_mean_count['rating_counts'] = pd.DataFrame(movie_data.groupby('title')['rating'].count())

print(ratings_mean_count.sort_values('rating_counts', ascending=False).head())

In [ ]:
user_movie_rating = movie_data.pivot_table(index='userId', columns='title', values='rating')

# Filling NaNs with 0 for cosine similarity calculation
user_movie_rating.fillna(0, inplace=True)

print(user_movie_rating.head())

### Experiment
- **What are we changing?** We are treating each movie as a vector in an N-dimensional space (where N is the number of users), and using `NearestNeighbors` with the `cosine` metric to find the 5 closest vectors.
- **What do we expect to happen?** Movies with similar user rating patterns should mathematically cluster close to each other, outputting sensible recommendations (e.g., sci-fi fans rating *The Matrix* highly might also rate similar movies highly).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

movie_features = user_movie_rating.T

knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6)
knn.fit(movie_features)

print("kNN Model fitted successfully.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target_movie = 'Matrix, The (1999)'
target_idx = movie_features.index.get_loc(target_movie)

distances, indices = knn.kneighbors(movie_features.iloc[target_idx, :].values.reshape(1, -1))

similar_movies = []
similar_distances = []

for i in range(1, len(distances.flatten())):
    similar_movies.append(movie_features.index[indices.flatten()[i]])
    similar_distances.append(1 - distances.flatten()[i]) # 1 - distance = similarity

similarity_df = pd.DataFrame({
    'Movie': similar_movies,
    'Cosine Similarity': similar_distances
})

print(f"Top 5 movies similar to {target_movie}:")
print(similarity_df)

### Visualization Interpretation
- **What pattern is visible?** A horizontal bar chart of Cosine Similarity scores for the top 5 nearest neighbors to the target movie.
- **What should the reader learn?** A similarity score of 1.0 means the vectors are perfectly aligned. We expect the top recommendations to have scores close to 1.0.

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    x='Cosine Similarity',
    y='Movie',
    data=similarity_df,
    palette='viridis'
)

plt.title(f'Top 5 Movies Similar to {target_movie}', fontsize=14, fontweight='bold')
plt.xlabel('Cosine Similarity', fontsize=12)
plt.ylabel('Movie', fontsize=12)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Visualization Analysis: Cosine Similarity
- **Marking Values:** The longest bars represent movies that are mathematically closest in angle to the target movie.
- **Correct Interpretation:** Cosine similarity ignores the magnitude of the vectors (e.g., how many total ratings a movie has) and strictly evaluates the pattern (angle). If User A rates two movies 5 stars, and User B rates them both 1 star, the angle between the movies is perfectly aligned.
- **Common Mistakes:** Confusing Cosine Similarity with Euclidean Distance. In a massive, sparse matrix (thousands of users, mostly 0s), Euclidean distance fails due to the curse of dimensionality, while Cosine Similarity thrives.

### Common Mistakes
- **Beginner Mistake:** Using Euclidean distance instead of Cosine distance for sparse recommendation matrices. Also, forgetting to handle `NaN` values (untouched movies) properly — filling them with 0 is necessary here so the math functions properly.

### Practical Takeaway
- **Industry Application:** This is the foundational logic behind e-commerce and streaming recommendation engines (e.g., "Because you watched The Matrix, you might like...").

### Key Insight
- **Memorable lesson:** kNN doesn't just "classify" data — it's fundamentally a spatial lookup tool. By changing our metric from Euclidean to Cosine, we instantly transform a classification algorithm into a powerful recommendation engine.